# Tugas Pengganti Kuliah 19 Oktober 2026: 
# `TensorFlow`-2: Model Prediksi Neural Networks

* Nama :  Muhammad Bani Rifqii Surasantika
* Nim  :  101022300206
* Kelas:  BS1TE-47-REG-01

> **INSTURKSI**
>
> * File tugas ini, isinya hampir sama dengan modul DL-Fiky-3-2.pdf yang telah diberikan, tapi tanpa kode-kode python
> * Bacalah modul DL-Fiky-2 dan DL-Fiky-3-2.pdf yang telah diberikan secara seksama dan teliti
> * Setiap program yang ada di modul tersebut harap dicoba dieksekusi di file ini sesuai dengan bagian-bagiannya
> * Pastikan luaran hasil eksekusi tidak ada error
> * Apabila telah selesai dieksekusi dan tidak ada error di luaran, lengkapi Nama, Nim, Kelas, kemudian upload file tugas ini ke LMS di bagian Assessment: Tugas 2.
> * Batas pengumpulan terakhir adalah **Sabtu, 14 November 2026, jam 18.00.**

### **A. `TensorFlow` Keras API (`tf.keras`)**

#### **A.1. Membangun Model Regresi Linier**

* Kita akan membuat model sederhana untuk memecahkan masalah regresi linier. Pertama kita akan buat sebuah dataset mainan pada `NumPy` dan memberikan visualisasi untuk dataset tersebut.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
X_train = np.arange(10).reshape((10, 1)) # 10 data fitur 1 dimensi
y_train = np.array([1.0, 1.3, 3.1, 2.0, 5.0, 6.3, 6.6, 7.4, 8.0, 9.0]) # 10 data␣
→vektor target 1 dimensi
plt.plot(X_train, y_train, 'o', markersize=10)
plt.xlabel('x')
plt.ylabel('y')
plt.show()

* Kita akan standarisasi fitur-fitur (*mean centering* dan membaginya dengan standar deviasi), kemudian membuat dataset `TensorFlow`.

In [ ]:
X_train_norm = (X_train- np.mean(X_train))/np.std(X_train)
ds_train_orig = tf.data.Dataset.from_tensor_slices(
(tf.cast(X_train_norm, tf.float32),
tf.cast(y_train, tf.float32)))

Kemudian, kita dapat mendefinisikan model untuk regresi linier sebagai $z = wx+b$. Di sini kita akan menggunakan API dari `Keras`. `tf.keras` menyediakan layer-layer yang telah didefinisikan sebelumnya (*predefined layers*) untuk membangun model NN yang kompleks, tetapi untuk memulai kita akan mempelajari bagaimana mendefinisikan sebuah model dari awal (*from scratch*). Nanti akan kita lihat bagaimana menggunakan *predefined layers* ini. 

Untuk masalah regresi ini, kita akan definisikan sebuah class yang berasal dari `tf.keras.Model`. *Subclassing* dengan `tf.keras.Model` membuat kita dapat menggunakan tool-tool `Keras` untuk mengeksplorasi model, training, dan evaluasi. Pada konstruktor class, kita akan definisikan parameter-parameter dari model, yaitu $w$ adalah parameter bobot dan $b$ adalah parameter bias. Kemudian kita akan memanggil metode `.call()`untuk menentukan bagaimana model ini menggunakan data input dalam men-*generate* output-outputnya. 

In [ ]:
class MyModel(tf.keras.Model):
    def __init__(self):
        super(MyModel, self).__init__()
        self.w = tf.Variable(0.0, name='weight')
        self.b = tf.Variable(0.0, name='bias')

def call(self, x):
    return self.w*x + self.b

* Selanjutnya kita akan membuat (*instantiate*) model baru dari class `MyModel()` yang bisa di-training berdasarkan data training. `Keras` API dari `TensorFlow` menyediakan sebuah metode `.summary()` untuk model-model yang dibuat dari `tf.keras.Model`, yang dapat digunakan untuk mendapatkan ringkasan (*summary*) dari komponen-komponen model layer demi layer dan jumlah parameter tiap layer. Karena kita telah membuat sub-class model kita dari `tf.keras.Model`, metode `.summary()` tersedia untuk kita gunakan juga. Tetapi, untuk dapat memanggil `model.summary()`, kita perlu mendefinisikan dimensi input (jumlah features) pada model ini dengan memanggil `model.build()` dengan *shape* data input yang diharapkan.

In [ ]:
model = MyModel()
model.build(input_shape=(None, 1))
model.summary()

Terlihat bahwa kita menggunakan `None` sepagai sebuah *placeholder* untuk dimensi pertama dari input tensor yang diharapkan melalui `model.build` sehingga kita dapat menggunakan ukuran batch sembarang. Tetapi, jumlah feature adalah tetap (di sini 1) yang sesuai dengan jumlah parameter bobot dari model. Membuat layer-layer dan parameter-parameter model setelah *instantiation*, dengan memanggil metode `.build()` disebut dengan `late variable creation`. Untuk model sederhana ini, kita telah membuat parameter-parameter model di dalam konstruktor, sehingga menyatakan `input_shape` via `.build()` tidak ada efek lebih jauh terhadap parameter-parameter ini. Tetapi tetap diperlukan jika kita ingin memanggil `model.summary`. 

Setelah mendefinisikan model, kita dapat mendefinisikan *cost function* yang kita ingin minimalkan untuk mencari pembobot optimal. Di sini, akan kita gunakan *mean squared error* (MSE) sebagai cost function. Lebih jauh, untuk *learn* parameter-parameter model, kita akan gunaan *Stochastic Gradient Descent* (SGD). Pada bagian ini, kita akan mengimplementasikan sendiri training dengan prosedur SGD. Tetapi, pada bagian selanjutnya kita akan menggunakan metode-metode `Keras` yaitu `compile()` dan `fit()` untuk melakukan hal yang sama. 

* Untuk mengimplementasikan algoritma SGD, kita perlu menghitung gradien-gradien. Kita tidak akan menghitung secara manual gradien-gradien tersebut, tetapi akan menggunakan API `TensorFlow` yaitu `tf.GradientTape` (akan dijelaskan pada bab selanjutnya).

In [ ]:
def loss_fn(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true- y_pred))
## testing the function:
yt = tf.convert_to_tensor([1.0])
yp = tf.convert_to_tensor([1.5])

loss_fn(yt, yp)

In [121]:
def train(model, inputs, outputs, learning_rate):
    with tf.GradientTape() as tape:
        current_loss = loss_fn(model(inputs), outputs)
    dW, db = tape.gradient(current_loss, [model.w, model.b])
    model.w.assign_sub(learning_rate * dW)
    model.b.assign_sub(learning_rate * db)

* Selanjutnya, akan kita set hyperparameter dan men-training model untuk 200 epoch. Kita akan membuat versi batch dari dataset dan memanggil metode `.repeat` dataset dengan `count=None`, yang akan menghasilkan pengulangan dataset tanpa batas. 

In [ ]:
def train(model, inputs, outputs, learning_rate):
with tf.GradientTape() as tape:
        current_loss = loss_fn(model(inputs), outputs)
    dW, db = tape.gradient(current_loss, [model.w, model.b])
    model.w.assign_sub(learning_rate * dW)
    model.b.assign_sub(learning_rate * db)

In [ ]:
tf.random.set_seed(1)

num_epochs = 200
log_steps = 100
learning_rate = 0.001
batch_size = 1
steps_per_epoch = int(np.ceil(len(y_train) / batch_size))

ds_train = ds_train_orig.shuffle(buffer_size=len(y_train))
ds_train = ds_train.repeat(count=None)
ds_train = ds_train.batch(1)

Ws, bs = [], []

for i, batch in enumerate(ds_train):
    if i >= steps_per_epoch * num_epochs:
        break
    Ws.append(model.w.numpy())
    bs.append(model.b.numpy())
    
    bx, by = batch
    loss_val = loss_fn(model(bx), by)
    
    train(model, bx, by, learning_rate=learning_rate)
    if i % log_steps == 0:
        print('Epoch {:4d} Step {:2d} Loss {:6.4f}'.format(
            int(i / steps_per_epoch), i, loss_val))

* Kita akan melihat model yang telah ditraining dan melakukan *plotting*. Untuk data tes, kita akan membuat dengan array `NumPy` dengan harga-harga dengan jarak sama antara $0-9$. Karena kita telah men-training model dengan features yang terstandarisasi (*standardized*), kita akan mengimplementasikan standarisasi yang sama pada data test.

In [ ]:
print('Final Parameters:', model.w.numpy(), model.b.numpy())

X_test = np.linspace(0, 9, num=100).reshape(-1, 1)
X_test_norm = (X_test - np.mean(X_train)) / np.std(X_train)
y_pred = model(tf.cast(X_test_norm, dtype=tf.float32))

fig = plt.figure(figsize=(13, 5))
ax = fig.add_subplot(1, 2, 1)
plt.plot(X_train_norm, y_train, 'o', markersize=10)
plt.plot(X_test_norm, y_pred, '--', lw=3)
plt.legend(['Training examples', 'Linear Reg.'], fontsize=15)
ax.set_xlabel('x', size=15)
ax.set_ylabel('y', size=15)
ax.tick_params(axis='both', which='major', labelsize=15)

ax = fig.add_subplot(1, 2, 2)
plt.plot(Ws, lw=3)
plt.plot(bs, lw=3)
plt.legend(['Weight w', 'Bias unit b'], fontsize=15)
ax.set_xlabel('Iteration', size=15)
ax.set_ylabel('Value', size=15)
ax.tick_params(axis='both', which='major', labelsize=15)

plt.show()

Gambar yang dihasilkan setelah mengeksekusi kode-kode terakhir, di sebelah kiri menunjukan *scatterplot* dari data training dan model regresi linier yang telah dihasilkan. Sedangkan di sebelah kanan menunjukan proses konvergen dari pembobot $w$ dan unit bias $b$.

#### **A.2. Training Model via `.compile()` dan `.fit()`**

Pada contoh sebelumnya, kita telah mempelajari bagaimana melakukan training sebuah model dengan menuliskan fungsi `train()`, dan mengimplementasikan metode optimasi SGD. Tetapi, menuliskan fungsi `train()` bisa menjadi pekerjaan berulang untuk projek-projek berbeda. Keras API pada `TensorFlow` telah menyediakan metode training yaitu `.fit()` yang dapat dipanggil pada model yang telah dibentuk (*instantiated*). Untuk melihat bagaimana hal ini bekerja, kita akan membuat model dan mengkompilasinya dengan memilih *optimizer*, *loss function* dan metrik evaluasi.

In [ ]:
tf.random.set_seed(1)
model = MyModel()
model.compile(optimizer='sgd',
              loss=loss_fn,
              metrics=['mae', 'mse'])



* Sekarang dapat kita panggil metode `fit()` untuk men-training model. Kita bisa melewatkan dataset yang telah di-*batch* (seperti `ds_train` yang telah dibuat di contoh sebelumnya). Tetapi kali ini, kita bisa melewatkan array `NumPy` untuk $x$ dan $y$ secara langsung tanpa harus membuat sebuah dataset. 

In [ ]:
model.fit(X_train_norm, y_train,
          epochs=num_epochs, batch_size=batch_size,
          verbose=1)

* Setelah model di-training seperti di atas, kita dapat memvisualisasikan hasilnya dan pastikan bahwa hasilnya sama dengan metode sebelunya.

In [ ]:
print(model.w.numpy(), model.b.numpy())
X_test = np.linspace(0, 9, num=100).reshape(-1, 1)
X_test_norm = (X_test- np.mean(X_train)) / np.std(X_train)
y_pred = model(tf.cast(X_test_norm, dtype=tf.float32))
fig = plt.figure(figsize=(13, 5))
ax = fig.add_subplot(1, 2, 1)
plt.plot(X_train_norm, y_train, 'o', markersize=10)
plt.plot(X_test_norm, y_pred, '--', lw=3)
plt.legend(['Training Samples', 'Linear Regression'], fontsize=15)
ax = fig.add_subplot(1, 2, 2)
plt.plot(Ws, lw=3)
plt.plot(bs, lw=3)
plt.legend(['W', 'bias'], fontsize=15)
plt.show()

#### **A.3. Membangun Multilayer Perceptron untuk Mengklasifikasikan Bunga pada Dataset Iris**

Pada contoh sebelumnya, kita telah melihat bagaimana membuat sebuah model dari awal dan model di-training menggunakan optimasi SGD. Meskipun kita mulai dengan contoh paling sederhana, kita dapat melihat bahwa mendefinisikan model dari awal tidak menarik dan tidak praktis. `TensorFlow` sebetulnya menyediakan layer-layer yang telah terdefinisi melalui `tf.keras.layers` yang dapat digunakan untuk blok bangunan (*building blocks*) dari model NN. Pada bagian ini, kita akan mempelajari bagaimana menggunakan layer-layer ini dalam memecahkan permasalahan klasifikasi menggunakan dataset Iris dan membangun perceptron dua layer menggunakan `Keras` API. 

* Kita mulai dengan mendapatkan data dari`tensorflow_datasets`.

In [ ]:
import tensorflow_datasets as tfds
iris, iris_info = tfds.load('iris', with_info=True)
print(iris_info)

Informasi hasil eksekusi kode di atas menunjukan bahwa dataset ini hanya mempunyai satu partisi, sehingga kita harus memisahkan dataset tersebut ke dalam partisi training dan testing. Kita asumsikan bahwa dua pertiga dari dataset akan digunakan sebagai training dan sisanya untuk testing. Librari `tensorflow_datasets` menyediakan tool yang baik untuk menentukan *slices* dan *splits* via objek `DatasetBuilder` sebelum *loading* dataset. Lebih jauh menyangkut *splits* dapat ditemukan di link berikut: [https://www.tensorflow.org/datasets/splits](https://www.tensorflow.org/datasets/splits).

Pendekatan lain adalah dengan pertama kali me-*load* keseluruhan dataset, kemudian gunakan `.take()` dan `.skip()` untuk men-*split* dataset ke dalam dua partisi. Jika dataset tersebut tidak di-*shuffle* saat pertama kali, kita juga bisa melakukan *shuffle*. Tetapi, kita harus hati-hati karena bisa jadi akan mencampur data testing dan training yang harus dihindari pada machine learning. Untuk menghindarinya, kita harus melakukan *setting* argumen `reshuffle_each_iteration=False`, pada metode `.shuffle()`. 

* Kode untuk *splitting* dataset ke dalam training dan test sebagai berikut.

In [ ]:
tf.random.set_seed(1)
ds_orig = iris['train']
ds_orig = ds_orig.shuffle(150, reshuffle_each_iteration=False)
print(next(iter(ds_orig)))
ds_train_orig = ds_orig.take(100)
ds_test = ds_orig.skip(100)

* Melihat jumlah *examples* pada training dan testing.

In [ ]:
## checking the number of examples:
n = 0
for example in ds_train_orig:
n += 1
print(n)
n = 0
for example in ds_test:
9
n += 1
print(n)

* Selanjutnya, kita telah melihatnya pada bagian sebelumnya, yaitu kita perlu menerapkan transformasi via  metode `.map()` untuk merubah dictionary ke sebuah tupple. 

In [ ]:
ds_train_orig = ds_train_orig.map(
    lambda x: (x['features'], x['label']))
ds_test = ds_test.map(
    lambda x: (x['features'], x['label']))
next(iter(ds_train_orig))

Sekarang kita sudah siap untuk menggunakan API `Keras` untuk membangun model secara efisien. Khususnya, dengan menggunakan class `tf.keras.Sequential` kita dapat menumpuk beberapa layer-layer `Keras` yang sudah tersedia di link berikut [https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers](https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers). Di sini kita akan menggunakan layer `Dense` yaitu `tf.keras.layers.Dense` yang juga dikenal sebagai *fully connected layer* atau *linear layer* dengan representasi terbaik menggunakan $f\left(w \times x + b\right)$. Dimana $x$ adalah features input, $w$ dan $b$ adalah matriks pembobot dan vektor bias, dan $f$ menyatakan fungsi aktivasi. 

Masing-masing layer pada NN menerima input-input dari layer sebelumnya, oleh sebab itu dimensinya (*rank* dan *shape*) adalah tetap. Biasanya, kita perlu memperhatikan dimensi output hanya ketika kita mendesain arsitektur NN. Pada contoh ini, kita ingin mendefinisikan model dengan dua hidden layer, dimana yang pertama menerima input dari empat features dan memproyeksikannya ke dalam 16 neurons. Hidden layer kedua menerima input dari layer sebelumnya (dengan ukuran 16) dan memproyeksikannya ke dalam tiga neuron output karena kita mempunyai label tiga class. 

* Model tersebut bisa kita buat dengan menggunakan class `Sequential` dan layer `Dense` pada `Keras menggunakan kode berikut.

In [ ]:
iris_model = tf.keras.Sequential([
tf.keras.layers.Dense(16, activation='sigmoid',
name='fc1', input_shape=(4,)),
tf.keras.layers.Dense(3, name='fc2', activation='softmax')])
iris_model.summary()

Dapat kita perhatikan dari kode di atas, *shape* dari input pada layer pertama ditentukan via `input_shape=(4,)`, oleh sebab itu kita tidak perlu memanggil fungai `.build()` lagi untuk menggunakan `iris_model_summary()`.

*Summary* dari model mengindikasikan bahwa layer pertama (`fc1`) mempunyai 80 parameter, dan layer kedua mempunyai 51 parameter. Kita dapat memverifikasi jumlah tersebut dengan $(n_{in}+1) \times n_{out}$, dimana $n_{in}$ adalah jumlah unit input, dan $n_{out}$ adalah jumlah unit output. Pada *fully (densely) connected layer*, parameter yang harus ditentukan saat training adalah matriks pembobot dengan ukuran $n_{in} \times n_{out}$ dan vektor bias dengan ukuran $n_{out}$. Lebih jauh, kita gunakan fungsi aktivasi sigmoid untuk layer pertama dan fungsi aktivasi softmax untuk layer output. Fungsi aktivasi softmax pada layer terakhir digunakan untuk mendukung klasifikasi multi-class karena kita mempunyai tiga label class pada luaran. Fungsi-fungsi aktivasi yang lain akan dibahas di bagian selanjutnya pada bab ini.

* Kita akan *compile* model ini dan menspesifikasikan *loss function*, *optimizer* dan juga metriks untuk evaluasi.

In [ ]:
iris_model.compile(optimizer='adam',
loss='sparse_categorical_crossentropy',
metrics=['accuracy'])

num_epochs = 100
training_size = 100
batch_size = 2
steps_per_epoch = np.ceil(training_size / batch_size)
ds_train = ds_train_orig.shuffle(buffer_size=training_size)
ds_train = ds_train.repeat()
ds_train = ds_train.batch(batch_size=batch_size)
ds_train = ds_train.prefetch(buffer_size=1000)
history = iris_model.fit(ds_train, epochs=num_epochs,
steps_per_epoch=steps_per_epoch,
verbose=0)

*Returned variable* `history` menjaga *training loss* dan *training accuracy* (karena dispesifikasikan sebagai metriks pada `iris_model.compile()`) pada setiap epoch. 

* Kita akan menggunakan kode berikut untuk memvisualisasikan *learning curve*.

In [ ]:
hist = history.history
fig = plt.figure(figsize=(12, 5))
ax = fig.add_subplot(1, 2, 1)
ax.plot(hist['loss'], lw=3)
ax.set_title('Training loss', size=15)
ax.set_xlabel('Epoch', size=15)
ax.tick_params(axis='both', which='major', labelsize=15)
ax = fig.add_subplot(1, 2, 2)
ax.plot(hist['accuracy'], lw=3)
ax.set_title('Training accuracy', size=15)
ax.set_xlabel('Epoch', size=15)
ax.tick_params(axis='both', which='major', labelsize=15)
plt.tight_layout()
#plt.savefig('ch13-cls-learning-curve.pdf')
plt.show()

#### **A.4. Evaluasi Model Hasil Training dengan Dataset Test**

* Karena kita telah menspesifikasikan `accuracy` sebagai metriks evaluasi pada `iris_model.compile()`, sekarang ktia dapat langsung mengevaluasi model pada dataset testing.

In [ ]:
results = iris_model.evaluate(ds_test.batch(50), verbose=0)
print('Test loss: {:.4f} Test Acc.: {:.4f}'.format(*results))

Dapat kita perhatikan bahwa kita perlu untuk mem-*batch*-kan juga dataset testing untuk menjamin bahwa input pada model mempunyai dimensi yang tepat (rank). Seperti yang telah kita diskusikan sebelumnya, memanggil metode `.batch()` akan meningkatkan rank dari tensor yang diperoleh (rank bertambah $1$). Data input untuk metode `.evaluate()` harus mempunyai satu dimensi yang telah ditentukan untuk batch, meskipun di sini ukuran batch tidak menjadi masalah. Oleh karena itu, jika kita melewatkan `ds_batch.batch(50)` pada metode `.evaluate()`, keseluruhan dataset testing akan diproses dengan batch berukuran $50$. Tetapi jika kita melewatkan `ds_batch.batch(1)` maka $50$ batch dengan ukuran $1$ akan diproses. 

#### **A.5. Menyimpan dan Reloading Model yang Telah Ditraining**

* Model yang telah ditraining dapat disimpan pada disk untuk digunakan selanjutnya dengan kode berikut.

In [ ]:
iris_model.save('iris-classifier.h5',
overwrite=True,
include_optimizer=True,
save_format='h5')

Opsi pertama adalah nama file. Memanggil metode `iris_model.save()` akan menyimpan keduanya, baik arsitektur model maupun parameter-parameter yang telah ditentukan dengan training. Tetapi, jika kita ingin menyimpan aristekturnya saja, maka dapat digunakan metode `iris_model.to_json()` yang akan menyimpan konfigurasi model dengan format `JSON`. Atau jika hanya ingin menyimpan pembobot-pembobot model, dapat kita gunakan `iris_model.save_weights()`. Sedangkan `save_format` dapa dispesifikasikan menjadi `h5` untuk format `HDF5` atau `tf` untuk format `TensorFlow`.

* Untuk me-reload model yang telah disimpan, dimana kita telah menyimpan arsitektur model dan juga pembobot, dan sekaligus memverifikasinya dengan melihat summary, maka dapat digunakan kode berikut.

In [ ]:
iris_model_new = tf.keras.models.load_model('iris-classifier.h5')
iris_model_new.summary()

* Selanjutnya, kita dapat mengevaluasi model baru yang telah di-*reload* menggunakan dataset testing untuk menverifikasi bahwa hasilnya sama dengan sebelumnya.

In [ ]:
results = iris_model_new.evaluate(ds_test.batch(50), verbose=0)
print('Test loss: {:.4f} Test Acc.: {:.4f}'.format(*results))

In [ ]:
labels_train = []
for i,item in enumerate(ds_train_orig):
labels_train.append(item[1].numpy())
labels_test = []
for i,item in enumerate(ds_test):
labels_test.append(item[1].numpy())
print('Training Set: ',len(labels_train), 'Test Set: ', len(labels_test))

In [ ]:
iris_model_new.to_json()

### **B. Memilih Fungsi Aktivasi untuk Multilayer Neural Networks**

Di bagian sebelumnya kita telah mempelajari fungsi aktivasi fungsi sigmoid (logistik) $\sigma(z)=\displaystyle\frac{1}{1+e^{-z}}$. Pada bagian ini kita akan mempelajari lebih jauh menyangkut fungsi-fungsi non-liniear lain yang berguna untuk mengimplementasikan NNs. Secara teknis kita dapat menggunakan fungsi sembarang sebagai fungsi aktivasi pada multilayer NN selama bisa diturunkan (*differentiable*). Bahkan kita bisa menggunakan fungsi linier seperti yang kita pakai di Adaline. Tetapi, penggunaan fungsi linier tidak berguna secara praktis, karena penjumlahan fungsi-fungsi linier akan menghasilkan fungsi linier juga. Non-linieritas diperkenalkan ke dalam ANN supaya bisa menghasilkan solusi untuk masalah-masalah kompleks. 

Fungsi aktivasi logistik yang digunakan pada bab sebelumnya mungkin yang paling dekat menyerupai konsep neuron di otak. Tetapi, fungsi aktivasi sigmoid mempunyai permasalahan jika input mempunyai nilai sangat negatif sehingga probabilitas mendekati nol. Jika fungsi sigmoid menghasilkan output yang dekat dengan nol, proses learning pada NN akan berjalan sangat lambat dan lebih mungkin terjebak pada minimum lokal. Dengan alasan inilah fungsi tangen hiperbolik lebih banyak dipilih sebagai fungsi aktivasi pada hidden layer. 

Sebelum kita mendiskusikan seperti apa fungsi tangen hiperbolik, rekapitulasi dari fungsi logistik akan dijelaskan kembali secara singkat. Generalisasi dari fungsi tersebut agar lebih berguna untuk masalah-masalah klasifikasi multi-class akan dijelaskan kemudian.

#### **B.1. Fungsi Logistik**

Fungsi logistik merupakan kasus spesial dari fungsi sigmoid. Dari bab sebelumnya, fungsi logistik dapat digunakan untuk memodelkan probabilitas bahwa sebuah sampel $x$ milik dari class positif pada kasus *binary classification*. 

Net input $z$ merupakan kombinasi linier dari input-input dengan persamaan berikut ini.

$$z = w_0 x_0 + w_1 x_1+\dots+w_m x_m = \sum_{i=0}^m w_ix_i=\mathbf{w}^T\mathbf{x}$$

Fungsi logistik akan menghitung kuantitas berikut:

$$\phi_{logistik}(z)=\displaystyle\frac{1}{1+e^{-z}}$$

Sebagai catatan $w_0$ adalah unit bias dengan $x_0=1$. 

* Untuk melihat hal yang lebih konkrit, kita akan membuat sebuah model untuk data dua dimensi $x$ dan sebuah model untuk vektor pembobot $\mathbf{w}$.

In [ ]:
import numpy as np
X = np.array([1, 1.4, 2.5]) ## first value must be 1
w = np.array([0.4, 0.3, 0.5])
def net_input(X, w):
return np.dot(X, w)
def logistic(z):
return 1.0 / (1.0 + np.exp(-z))
def logistic_activation(X, w):
z = net_input(X, w)
return logistic(z)
print('P(y=1|x) = %.3f' % logistic_activation(X, w))

Jika kita hitung net input ($z$) dan menggunakannya untuk mengaktivasi sebuah neuron logistik dengan harga-harga feature dan koefisien pembobot seperti pada kode di atas, maka kita akan mendapatkan harga $0.88$. Dari hasil tersebut dapat diinterpretasikan bahwa probabilitas bahwa $x$ berasal dari positif class adalah $0.88$ atau $88.8\%$.

* Dari bab sebelumnya kita gunakan teknik *one-hot encoding* untuk merepresentasikan label-label *ground truth* dari multi-class dan mendesain layer output yang terdiri dari beberapa unit aktivasi logistik. Tetapi, seperti yang terlihat pada kode berikut, layer output dengan unit aktivasi logistik tidak akan menghasilkan harga-harga probabilitas yang mempunyai arti atau dapat diinterpretasikan. 

In [ ]:
# W : array with shape = (n_output_units, n_hidden_units+1)
# note that the first column are the bias units
W = np.array([[1.1, 1.2, 0.8, 0.4],
[0.2, 0.4, 1.0, 0.2],
[0.6, 1.5, 1.2, 0.7]])
# A : data array with shape = (n_hidden_units + 1, n_samples)
# note that the first column of this array must be 1
A = np.array([[1, 0.1, 0.4, 0.6]])
Z = np.dot(W, A[0])
y_probas = logistic(Z)
print('Net Input: \n', Z)
print('Output Units:\n', y_probas)

Seperti yang kita lihat pada output, harga-harga tersebut tidak dapat diinterpretasikan sebagai harga probabilitas untuk masalah klasifikasi dengan tiga class, karena ketika dijumlahkan tidak akan sama dengan $1$. Tetapi, hal ini tidak menjadi masalah, jika kita menggunakan model hanya untuk meprediksi label class dan bukan probabilitas keanggotaan dari class. 

* Cara untuk melakukan prediksi label class dari unit output yang diperoleh sebelumnya dapat menggunakan harga maksimum seperti berikut.

In [ ]:
y_class = np.argmax(Z, axis=0)
print('Predicted class label: %d' % y_class)

#### **B.2. Estimasi Probabilitas Class dari Klasifikasi Multi-Class via Fungsi Softmax**

Pada bagian sebelumnya kita telah melihat bagaimana kita memperoleh label class dengan menggunakan fungsi `argmax`. Di bab sebelumnya, kita juga menggunakan fungsi aktivasi `activation=softmax` pada layer terakhir model MLP. Fungsi `softmax` merupakan bentuk *soft* dari fungsi `argmax`, dimana fungsi `softmax` memberikan hasil probabilitas tiap class bukan indeks class. Dengan demikian, fungsi tersebut dapat digunakan untuk menghitung probabilitas keanggotaan dari class (jumlah keseluruhan menjadi $1$) pada skenario klasifikasi multi-class (*multinomial logistic regression*).

Pada `softmax`, probabilitas untuk sampel tertentu dengan net input $z$ berasal dari class ke-$i$, dapat dihitung dengan normaliasi pada penyebut (*denominator*) dengan penjumlahan fungsi linier yang diboboboti secara eksponensial.

$$p(z)=\phi(z)=\displaystyle\frac{e^{Z_i}}{\sum_{j=i}^M e^{Z_j}}$$

* Berikut kode program `softmax` sebagai contoh.

In [ ]:
def softmax(z):
return np.exp(z) / np.sum(np.exp(z))
y_probas = softmax(Z)
print('Probabilities:\n', y_probas)
np.sum(y_probas)

Dapat dilihat pada hasil kode di atas, probabilitas class-class terprediksi apabila dijumlahkan akan sama dengan $1$ seperti yang kita harapkan. Dapat dilihat juga bahwa prediksi label class hasilnya sama dengan hasil dari fungsi `argmax` dari output logistik (probabilitas terbesar adalah class $0$).

Dengan demikian, ketika kita akan membangun model klasifikasi multi-class pada `TensorFlow`, maka kita dapat menggunakan fungsi `tf.keras.activations.softmax()` untuk mengestimasi masing-masing probabilitas keanggotaan class. Untuk melihat bagaimana kita menggunakan fungsi aktivasi `softmax` pada `TensorFlow`, kode berikut merubah $z$ menjadi sebuah tensor, dengan tambahan dimensi yang dipesan untuk *batch size*.

In [ ]:
import tensorflow as tf
Z_tensor = tf.expand_dims(Z, axis=0)
tf.keras.activations.softmax(Z_tensor)

#### **B.3. Fungsi Tangen Hiperbolik *(a Hyperbolic Tangent)***

Fungsi sigmoid lain yang sering digunakan pada hidden layer dari ANNs adalah fungsi hiperbolik, atau biasanya dikenal dengan `tanh`, yang bisa diinterpretasikan sebagai versi penskalaan (*rescaled version*) dari fungsi logistik.

$$\begin{align}
\phi_{logistik} & = \displaystyle \frac{1}{1+e^{-z}}\nonumber\\
\phi_{tanh} & = 2 \times x \phi_{logistik} - 1 = \displaystyle \frac{e^Z - e^{-Z}}{e^Z+e^{-Z}}
\end{align}$$

Keuntungan dari tangen hiperbolik dibandingkan dengan fungsi logistik adalah tangen hiperbolik mempunyai spektrum output yang lebih lebar dengan range atara $(-1,1)$ yang dapat memperbaiki konvergensi dari algoritma backpropagation. Berbeda dengan fungsi logistik yang memberikan hasil output pada interval $(0,1)$

* Untuk membandingkan fungsi logistik dan tangen hiperbolik, berikut adalah plotting dari kedua fungsi sigmoid tersebut.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
def tanh(z):
e_p = np.exp(z)
e_m = np.exp(-z)
return (e_p- e_m) / (e_p + e_m)
z = np.arange(-5, 5, 0.005)
log_act = logistic(z)
tanh_act = tanh(z)
plt.ylim([-1.5, 1.5])
plt.xlabel('Net input $z$')
plt.ylabel('Activation $\phi(z)$')
plt.axhline(1, color='black', linestyle=':')
plt.axhline(0.5, color='black', linestyle=':')
plt.axhline(0, color='black', linestyle=':')
plt.axhline(-0.5, color='black', linestyle=':')
18
plt.axhline(-1, color='black', linestyle=':')
plt.plot(z, tanh_act,
linewidth=3, linestyle='--',
label='Tanh')
plt.plot(z, log_act,
linewidth=3,
label='Logistic')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

Dapat kita lihat di hasil gambar bahwa kedua fungsi sigmoid tersebut cukup serupa, tetapi fungsi `tanh` mempunyai ruang dua kali lebih besar dibandingkan dengan fungsi `logistic`. Secara alternatif, kita bisa menggunakan fungsi `tanh` dari `NumPy`.

In [ ]:
np.tanh(z)

* Sebagai opsi lain ketika membangun model NN, kita juga dapat menggunakan fungsi `tf.keras.activations.tanh()` dari `TensorFlow` untuk memperoleh hasil yang sama. 

In [ ]:
import tensorflow as tf
tf.keras.activations.tanh(z)

* Sebagai tambahan, fungsi logistik juga tersedia pada modul `special` dari [`SciPy`](https://scipy.org).

In [ ]:
from scipy.special import expit
expit(z)

* Kita juga bisa menggunakan fungsi `tf.keras.activations.sigmoid()` dari `TensorFlow` untuk menghitung hasil yang sama sebagai berikut.

In [ ]:
tf.keras.activations.sigmoid(z)

#### **B.4. Aktivasi dengan *Rectified Linear Unit* (Re-LU)**

*Rectified Linear Unit* (RELU) adalah fungsi aktivasi lain yang sering digunakan pada *Deep Neural Networks* (DNN). Sebelum membahas ReLU, kita harus memahami masalah *vanishing gradient problem* dari fungsi aktivasi `tanh` dan logistik.

Kita akan asumsikan bahwa awalnya kita mempunyai net input $z_1=20$ yang berubah ke $z_2=25$. Dengan menghitung aktivasi `tanh` kita akan peroleh $\phi(z_1)=1.0$ dan $\phi(z_2)=1.0$ yang tidak menghasilkan perubahan pada output (akibat perilaku asimptotik dari fungsi `tanh` dan error numerik). Artinya, turunan dari fungsi aktivasi ini terhadap net input $z$ akan semakin mengecil (menghilang) dengan $z$ semakin membesar. Oleh sebab itu, proses *learning* dari pembobot selama fase training akan sangat lambat karena bagian gradien akan semakin mendekati nol. Fungsi aktivasi ReLU memberikan solusi untuk masalah ini. Secara matematis ReLU didefinisikan sebagai berikut.

$$\phi(z)=max(0,z)$$

ReLU masih bersifat fungsi non-linier yang baik untuk *learning* fungsi-fungsi kompleks dengan NNs. Selain itu, turunan ReLU terhadap input selalu bernilai $1$ untuk input bernilai positif. Oleh sebab itu, ReLU dapat memecahkan masalah *vanishing gradient* sehingga cocok digunakan untuk DNN. 

* Pada `TensorFlow`, kita dapat menggunakan fungsi aktivasi ReLU dengan cara berikut.

In [ ]:
import tensorflow as tf
tf.keras.activations.relu(z)

Sampai sejauh ini kita telah mengetahui fungsi-fungsi aktivasi yang berbeda yang sering digunakan pada ANN. Sebagai tambahan informasi Gambar 3.2 berikut menunjukan fungsi-fungsi aktivasi yang akan digunakan pada bab-bab selanjutnya.

<center>
<img src="./images/ActivationFunction.png" alt="ForwardProp" width="550"/>
<div style="text-align: center">
<center>Gambar 3.2: Fungsi-fungsi aktivasi yang akan digunakan